In [1]:
# 使用Fiona打开Shapefile就像打开普通文件一样

import fiona
from fiona import crs

fp = r"C:\Users\tanzhenyu\Dataware\GeoPy\XiAn\XiAn.shp"
with fiona.open(fp) as fc:
    # 这里的fc是一个Collection对象
    print(f'fc是一个{type(fc)}对象')
    # 以PROJ4格式输出文件的投影信息
    print(f'文件采用{crs.to_string(fc.crs)}参考系统')
    # 输出要素的范围
    print(f'要素的地理范围为{fc.bounds}')
    # 输出要素集中要素的个数
    print(f'文件中包含{len(fc)}个要素')
    # fc中的每个元素都是一个要素（类型为dict），一般包含geometry和properties
    print(f"文件中第一个要素类型为{fc[0]['geometry']['type']}")
    # 属性值是一个OrderedDict
    print(f"文件中第一个要素包含的属性{fc[0]['properties']}")

fc是一个<class 'fiona.collection.Collection'>对象
文件采用+ellps=GRS80 +no_defs +proj=longlat参考系统
要素的地理范围为(107.65834737000003, 33.69600194000005, 109.82174724500004, 34.74398251500003)
文件中包含13个要素
文件中第一个要素类型为Polygon
文件中第一个要素包含的属性OrderedDict([('PAC', 610102), ('NAME', '新城区')])


In [2]:
with fiona.open(fp) as fc:
    for g in fc:
        # 这里的g是一个dict类型的变量
        print(g['properties']['NAME'])

新城区
碑林区
莲湖区
灞桥区
未央区
雁塔区
阎良区
临潼区
长安区
高陵区
鄠邑区
蓝田县
周至县


In [1]:
import pandas
import fiona
from fiona import crs
from shapely.geometry import Point, mapping

fn_in = "/Users/tanzhenyu/Dataware/GeoPy/DataForBook/CitiyPopulation.csv"
fn_out = '/Users/tanzhenyu/Dataware/GeoPy/DataForBook/CitiyPopulation.shp'

# 定义要写入数据的形式，比如几何体类型，非几何体属性类型
schema = {
    'geometry': 'Point',
    'properties': {
        'name': 'str',
        'population': 'int',
        'pop_density': 'float',
        'country': 'str'
    }
}

def row_to_shape(row, dst):
    point = Point(float(row['longitude']), float(row['latitude']))
    prop = {
        'name': row['name'],
        'population': int(row['population']),
        'pop_density': float(row['pop_density']),
        'country': row['pop_density']
    }
    dst.write({
        'geometry': mapping(point),
        'properties':prop}
    )

with fiona.open(fn_out, 'w', crs=crs.from_epsg(4326),
                driver='ESRI Shapefile', schema=schema) as dst:
    data = pandas.read_csv(fn_in, delimiter=',')
    # apply函数效率高于for循环
    data.apply(
        lambda row: row_to_shape(row, dst),
        axis=1
    )

    # 使用for循环进行处理
    # for _, row in data.iterrows():
    #     row_to_shape(row, dst)

In [14]:
import rasterio

fn = r"C:\Users\tanzhenyu\Dataware\GeoPy\DataForBook\XiAn-202108.tif"
with rasterio.open(fn) as ds:
    print('该栅格数据的基本数据集信息（这些信息都是以数据集属性的形式表示的）：')
    print(f'数据格式：{ds.driver}')
    print(f'波段数目：{ds.count}')
    print(f'影像宽度：{ds.width}')
    print(f'影像高度：{ds.height}')
    print(f'地理范围：{ds.bounds}')
    print(f'反射变换参数（六参数模型）：\n{ds.transform}')
    print(f'投影定义：{ds.crs}')
    # 获取第一个波段数据，跟GDAL一样索引从1开始
    # 直接获得numpy.ndarray类型的二维数组表示，如果read()函数不加参数，则得到所有波段（第一个维度是波段）
    band1 = ds.read(1)
    print(f'第一波段的最大值：{band1.max()}')
    print(f'第一波段的最小值：{band1.min()}')
    print(f'第一波段的平均值：{band1.mean()}')
    # 根据地理坐标得到行列号
    x, y = (ds.bounds.left + 300, ds.bounds.top - 300)  # 距离左上角东300米，南300米的投影坐标
    row, col = ds.index(x, y)  # 对应的行列号
    print(f'(投影坐标{x}, {y})对应的行列号是({row}, {col})')
    # 根据行列号得到地理坐标
    x, y = ds.xy(row, col)  # 中心点的坐标
    print(f'行列号({row}, {col})对应的中心投影坐标是({x}, {y})')
    # 那么如何得到对应点左上角的信息
    x, y =  ds.transform * (row, col)
    print(f'行列号({row}, {col})对应的左上角投影坐标是({x}, {y})')

该栅格数据的基本数据集信息（这些信息都是以数据集属性的形式表示的）：
数据格式：GTiff
波段数目：4
影像宽度：13831
影像高度：13201
地理范围：BoundingBox(left=143385.0, bottom=3554385.0, right=558315.0, top=3950415.0)
反射变换参数（六参数模型）：
| 30.00, 0.00, 143385.00|
| 0.00,-30.00, 3950415.00|
| 0.00, 0.00, 1.00|
投影定义：EPSG:32649
第一波段的最大值：65535
第一波段的最小值：0
第一波段的平均值：5804.964432258767
(投影坐标143685.0, 3950115.0)对应的行列号是(10, 10)
行列号(10, 10)对应的中心投影坐标是(143700.0, 3950100.0)
行列号(10, 10)对应的左上角投影坐标是(143685.0, 3950115.0)


In [15]:
# 将投影参考系转为地理参考系，然后另存为ERDAS的IMG格式进行存储
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio import crs

src_img = r"C:\Users\tanzhenyu\Dataware\GeoPy\DataForBook\XiAn-202108-AOI.tif"
dst_img = "XiAn-202108-AOI.img"

# 转为地理坐标系WGS84
dst_crs = crs.CRS.from_epsg('4326')
with rasterio.open(src_img) as src_ds:
    profile = src_ds.profile

    # 计算在新空间参考系下的仿射变换参数，图像尺寸
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_ds.crs, dst_crs, src_ds.width, src_ds.height, *src_ds.bounds)

    # 更新数据集的元数据信息
    profile.update({
        'driver': 'HFA',
        'crs': dst_crs,
        'transform': dst_transform,
        'width': dst_width,
        'height': dst_height,
        'nodata': 0
    })

    # 重投影并分波段写入数据
    with rasterio.open(dst_img, 'w', **profile) as dst_ds:
        for i in range(1, src_ds.count + 1):
            src_array = src_ds.read(i)
            dst_array = np.empty((dst_height, dst_width), dtype=profile['dtype'])

            reproject(
                # 源文件参数
                source=src_array,
                src_crs=src_ds.crs,
                src_transform=src_ds.transform,
                # 目标文件参数
                destination=dst_array,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                # 其它配置
                resampling=Resampling.cubic,
                num_threads=2)
            # 分波段写入
            dst_ds.write(dst_array, i)